# DMC Dreamer Core Comparison

This notebook trains and evaluates two Dreamer-style RSSM variants on DeepMind Control Suite:

- `size_debug_gru`: the regular block-GRU deterministic RSSM core.
- `size_debug_mamba3`: the experimental Mamba3 deterministic RSSM core.

The default run is intentionally small for Colab debugging. Increase `TRAIN_STEPS`, `ENV_NUM`, and `EVAL_EPISODES` for real benchmark runs.

## Colab / VSCode Notes

- In Colab, the first code cell mounts Google Drive and uses `/content/drive/MyDrive/Research/SSM_World_Models` as the persistent workspace.
- The repo is cloned to `Research/SSM_World_Models/r2dreamer` if the source folder is missing.
- Experiment logs are saved under `Research/SSM_World_Models/runs`, so they survive runtime resets.
- The experiment uses DMC proprio observations on `dmc_walker_walk`.
- Colab Python 3.12 can reject `pip install -e .[dmc]` because the repo declares Python `<3.12`; the setup cell installs third-party dependencies directly instead.
- The setup cell installs Mamba3 from `state-spaces/mamba` source if the Mamba3 import is missing.
- Use a GPU runtime for the Mamba3 variant; the source install builds CUDA extensions and can take several minutes.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Persistent Google Drive workspace. This is where Colab should keep the repo
# clone and experiment logs so they survive runtime resets.
DRIVE_WORKDIR = Path(os.environ.get(
    "SSM_WORLD_MODELS_DIR",
    "/content/drive/MyDrive/Research/SSM_World_Models",
))
AUTO_MOUNT_DRIVE = True


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


if AUTO_MOUNT_DRIVE and running_in_colab() and str(DRIVE_WORKDIR).startswith("/content/drive"):
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_WORKDIR.mkdir(parents=True, exist_ok=True)
    print("Using Drive workspace:", DRIVE_WORKDIR)

# Colab usually starts in /content with only the notebook present.
# If the r2dreamer source folder is not present, this cell can clone it.
REPO_URL = os.environ.get("R2DREAMER_REPO_URL", "https://github.com/hanswalker/r2dreamer.git")
BRANCH = os.environ.get("R2DREAMER_BRANCH", "mamba3-rssm-experiment")
AUTO_CLONE = True

# If auto-detection fails, set this to the folder that contains rssm.py.
# Examples:
# R2DREAMER_DIR = "/content/drive/MyDrive/Research/SSM_World_Models/r2dreamer"
# R2DREAMER_DIR = "/content/r2dreamer"
R2DREAMER_DIR = os.environ.get("R2DREAMER_DIR", "")
CLONE_DIR = Path(os.environ.get("R2DREAMER_CLONE_DIR", str(DRIVE_WORKDIR / "r2dreamer")))


def is_r2dreamer(path: Path) -> bool:
    return (path / "rssm.py").exists() and (path / "configs").is_dir()


def candidate_paths(start: Path):
    explicit = [Path(R2DREAMER_DIR).expanduser()] if R2DREAMER_DIR else []
    fixed = [
        DRIVE_WORKDIR / "r2dreamer",
        Path("/content/r2dreamer"),
        Path("/content/ssm-world/r2dreamer"),
        Path("/content/drive/MyDrive/Research/SSM_World_Models/r2dreamer"),
        Path("/workspace/ssm-world/r2dreamer"),
    ]
    for base in [*explicit, start, *start.parents, *fixed]:
        yield base
        yield base / "r2dreamer"
        yield base / "ssm-world" / "r2dreamer"

    for root in [start, DRIVE_WORKDIR, Path("/content"), Path("/workspace")]:
        if not root.exists():
            continue
        for pattern in ["r2dreamer", "*/r2dreamer", "*/*/r2dreamer", "*/*/*/r2dreamer"]:
            try:
                yield from root.glob(pattern)
            except OSError:
                pass


def find_r2dreamer(start: Path):
    seen = set()
    for candidate in candidate_paths(start):
        try:
            candidate = candidate.resolve()
        except OSError:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        if is_r2dreamer(candidate):
            return candidate
    return None


def clone_r2dreamer():
    CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    if CLONE_DIR.exists() and is_r2dreamer(CLONE_DIR):
        return CLONE_DIR.resolve()
    if CLONE_DIR.exists() and any(CLONE_DIR.iterdir()):
        raise FileExistsError(
            f"Clone target {CLONE_DIR} already exists and is not an r2dreamer source folder. "
            "Set R2DREAMER_CLONE_DIR to an empty path or delete the stale folder."
        )
    cmd = ["git", "clone", "--depth", "1"]
    if BRANCH:
        cmd += ["--branch", BRANCH]
    cmd += [REPO_URL, str(CLONE_DIR)]
    print("Cloning:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    return CLONE_DIR.resolve()


cwd = Path.cwd()
REPO_DIR = find_r2dreamer(cwd)
if REPO_DIR is None and AUTO_CLONE:
    REPO_DIR = clone_r2dreamer()
if REPO_DIR is None:
    raise FileNotFoundError(
        f"Could not find the r2dreamer source folder from cwd={cwd}. "
        "Clone/copy the repo first, or set R2DREAMER_DIR in this cell to the "
        "folder containing rssm.py and configs/."
    )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Using r2dreamer:", REPO_DIR)
print("Persistent workspace:", DRIVE_WORKDIR)


In [ ]:
import importlib
import importlib.util
import platform

# Fresh Colab setup switches. Leave these True for a new runtime.
INSTALL_DEPS = True
INSTALL_MAMBA3 = True

# Colab currently uses Python 3.12 in many runtimes. r2dreamer's pyproject declares
# Python >=3.11,<3.12, so `pip install -e .[dmc]` can fail before installing anything.
# Instead, this notebook installs the runtime dependencies directly and imports the
# checked-out source via sys.path, which was set in the repo-location cell.
RUNTIME_DEPS = [
    "torch==2.8.0",
    "torchrl==0.9.2",
    "tensordict==0.9.1",
    "ruamel.yaml==0.17.4",
    "moviepy==1.0.3",
    "einops==0.3.0",
    "numpy==1.26.0",
    "hydra-core==1.3.2",
    "gymnasium==1.2.0",
    "tensorboard>=2.20,<3",
    "setuptools==77.0.3",
    "dm_control==1.0.28",
    "mujoco==3.3.0",
    "matplotlib",
]

# Mamba3 currently needs the source install path from state-spaces/mamba.
# This can take a while because it builds CUDA extensions.
MAMBA3_REPO_URL = "git+https://github.com/state-spaces/mamba.git"


def run(cmd, *, env=None):
    print("Running:", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, env=env)


def module_exists(name):
    return importlib.util.find_spec(name) is not None


def mamba3_import_ok():
    try:
        from mamba_ssm.modules.mamba3 import Mamba3  # noqa: F401
        return True, None
    except Exception as exc:
        return False, exc


print("Python:", platform.python_version())

if INSTALL_DEPS:
    run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "wheel"])
    run([sys.executable, "-m", "pip", "install", *RUNTIME_DEPS])

if module_exists("torch"):
    import torch
else:
    raise RuntimeError("PyTorch is missing after dependency install. Check the pip output above.")

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("Warning: Mamba3 source builds are normally CUDA-oriented. Use a GPU runtime for the Mamba3 variant.")

ok, err = mamba3_import_ok()
if not ok and INSTALL_MAMBA3:
    # Official Mamba3 note: install from source with MAMBA_FORCE_BUILD=TRUE and no build isolation.
    env = os.environ.copy()
    env["MAMBA_FORCE_BUILD"] = "TRUE"
    run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--force-reinstall",
            MAMBA3_REPO_URL,
            "--no-build-isolation",
        ],
        env=env,
    )
    importlib.invalidate_caches()
    ok, err = mamba3_import_ok()

if importlib.util.find_spec("dm_control") is None:
    raise RuntimeError("dm_control is missing. Keep INSTALL_DEPS = True and rerun this cell.")

if not ok:
    raise RuntimeError(
        "Mamba3 is not importable, so the mamba3 variant cannot run. "
        "The expected import is `from mamba_ssm.modules.mamba3 import Mamba3`. "
        f"Last import error: {err!r}"
    )

print("dm_control import: ok")
print("Mamba3 import: ok")


In [ ]:
from datetime import datetime

# Keep these small for a first Colab run. For a real comparison, increase TRAIN_STEPS substantially.
DMC_TASK = "dmc_walker_walk"
TRAIN_STEPS = 3000
EVAL_EVERY = 1500
EVAL_EPISODES = 2
ENV_NUM = 4
BATCH_SIZE = 4
BATCH_LENGTH = 16
TRAIN_RATIO = 32
SEED = 0
REP_LOSS = "r2dreamer"

# Set to True to delete and rerun an existing logdir.
RESET_RUNS = False

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_ROOT = Path(os.environ.get("R2DREAMER_LOG_ROOT", str(DRIVE_WORKDIR / "runs" / "notebook_dmc_core_compare"))) / RUN_ID

VARIANTS = [
    {"name": "block_gru", "model": "size_debug_gru"},
    {"name": "mamba3", "model": "size_debug_mamba3"},
]

print("Task:", DMC_TASK)
print("Device:", DEVICE)
print("Log root:", LOG_ROOT)
print("Variants:", VARIANTS)

In [ ]:
import shutil
import time


def stream_command(cmd, env=None):
    print("Running:", " ".join(map(str, cmd)))
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)


def train_variant(variant):
    logdir = LOG_ROOT / variant["name"]
    if logdir.exists() and RESET_RUNS:
        shutil.rmtree(logdir)
    if (logdir / "latest.pt").exists() and not RESET_RUNS:
        print(f"Skipping {variant['name']} because {logdir / 'latest.pt'} exists.")
        return logdir

    env = os.environ.copy()
    env.setdefault("MUJOCO_GL", "egl")
    env.setdefault("PYOPENGL_PLATFORM", "egl")

    cmd = [
        sys.executable,
        "train.py",
        "env=dmc_proprio",
        f"env.task={DMC_TASK}",
        f"env.steps={TRAIN_STEPS}",
        f"env.env_num={ENV_NUM}",
        f"env.eval_episode_num={EVAL_EPISODES}",
        f"env.train_ratio={TRAIN_RATIO}",
        f"trainer.eval_every={EVAL_EVERY}",
        "trainer.update_log_every=500",
        f"model={variant['model']}",
        f"model.rep_loss={REP_LOSS}",
        "model.compile=False",
        f"device={DEVICE}",
        f"seed={SEED}",
        f"batch_size={BATCH_SIZE}",
        f"batch_length={BATCH_LENGTH}",
        "buffer.storage_device=cpu",
        "buffer.max_size=50000",
        f"logdir={logdir}",
    ]
    start = time.time()
    stream_command(cmd, env=env)
    print(f"Finished {variant['name']} in {(time.time() - start) / 60:.1f} minutes")
    return logdir


variant_logdirs = {}
for variant in VARIANTS:
    print("\n" + "=" * 80)
    print("Training/evaluating:", variant["name"])
    variant_logdirs[variant["name"]] = train_variant(variant)

variant_logdirs

In [ ]:
import json
import math


def read_metrics(logdir):
    path = Path(logdir) / "metrics.jsonl"
    if not path.exists():
        return None
    rows = []
    with path.open("r") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def series(rows, key):
    xs, ys = [], []
    for row in rows:
        if key in row:
            xs.append(row["step"])
            ys.append(row[key])
    return xs, ys


def available_metric_keys(rows):
    keys = set()
    for row in rows:
        keys.update(k for k in row.keys() if k != "step")
    return sorted(keys)


# If the training cell was skipped after a runtime reconnect, rebuild logdirs from LOG_ROOT.
if "variant_logdirs" not in globals():
    variant_logdirs = {variant["name"]: LOG_ROOT / variant["name"] for variant in VARIANTS}

all_metrics = {}
for name, logdir in variant_logdirs.items():
    rows = read_metrics(logdir)
    if rows is None:
        print(f"Warning: no metrics.jsonl for {name}: {Path(logdir) / 'metrics.jsonl'}")
        continue
    if not rows:
        print(f"Warning: empty metrics file for {name}: {Path(logdir) / 'metrics.jsonl'}")
        continue
    all_metrics[name] = rows

if not all_metrics:
    raise RuntimeError(
        "No metrics were found. Run the training cell first, or set LOG_ROOT to an existing run folder."
    )

summary = {}
for name, rows in all_metrics.items():
    eval_steps, eval_scores = series(rows, "episode/eval_score")
    train_steps, train_scores = series(rows, "episode/score")
    loss_steps, losses = series(rows, "train/opt/loss")
    if not eval_scores:
        print(f"Warning: {name} has no episode/eval_score entries.")
        print("Available metric keys:", available_metric_keys(rows))
    summary[name] = {
        "eval_count": len(eval_scores),
        "final_eval_step": eval_steps[-1] if eval_steps else math.nan,
        "final_eval_score": eval_scores[-1] if eval_scores else math.nan,
        "best_eval_score": max(eval_scores) if eval_scores else math.nan,
        "train_episode_count": len(train_scores),
        "final_train_score": train_scores[-1] if train_scores else math.nan,
        "final_train_loss": losses[-1] if losses else math.nan,
        "logdir": str(variant_logdirs[name]),
    }

for name, item in summary.items():
    print(
        f"{name}: eval_count={item['eval_count']}, "
        f"final_eval={item['final_eval_score']:.3f}, "
        f"best_eval={item['best_eval_score']:.3f}, "
        f"logdir={item['logdir']}"
    )

summary


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, rows in all_metrics.items():
    xs, ys = series(rows, "episode/eval_score")
    if xs:
        axes[0].plot(xs, ys, marker="o", label=name)
    xs, ys = series(rows, "episode/score")
    if xs:
        axes[1].plot(xs, ys, marker=".", label=name)

axes[0].set_title("Eval Score")
axes[0].set_xlabel("environment steps")
axes[0].set_ylabel("return")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_title("Train Episode Score")
axes[1].set_xlabel("environment steps")
axes[1].set_ylabel("return")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## Scaling Up

For a meaningful comparison, rerun with larger settings such as:

```python
TRAIN_STEPS = 100_000
EVAL_EVERY = 10_000
EVAL_EPISODES = 10
ENV_NUM = 16
BATCH_SIZE = 16
BATCH_LENGTH = 64
TRAIN_RATIO = 512
```

Keep all settings identical between variants except `model=size_debug_gru` versus `model=size_debug_mamba3`.